# Create SparkSession

In [2]:
from spark_utils import SparkUtils
from pathlib import Path
import shutil
import pyspark.sql.functions as F

kafka_connector = "org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.0"
su = SparkUtils("Structured Streaming Kafka", 
                "spark://spark-master:7077",
                spark_packages=kafka_connector)
su.spark


:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-db7b93f1-4ae6-40e4-b787-1bcd0caf1025;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.13;4.0.0 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.0.0 in central
	found org.apache.kafka#kafka-clients;3.9.0 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.7 in central
	found org.slf4j#slf4j-api;2.0.16 in central
	found org.apache.hadoop#hadoop-client-runtime;3.4.1 in central
	found org.apache.hadoop#hadoop-client-api;3.4.1 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.scala-lang.modules#scala-parallel-collections_2.13;1.2.0

## Custom producer

### Create `store-transactions` topic

```
    docker exec -it <Kafka container ID> \
     /opt/kafka/bin/kafka-topics.sh \
      --create --zookeeper zookeeper:2181 \
      --replication-factor 1 --partitions 1 \
      --topic store-transactions
```

### Run the producer

```
    docker exec -it <Spark-Notebook container ID> /bin/bash
    # cd src/producers/
    # python3 faker_dataset.py --broker kafka:9093 --topic store-transactions --records 20
```

### Run the consumer code

In [3]:
# Create the remote connection
store_transactions_df = (su.spark.readStream
            .format("kafka")
            .option("kafka.bootstrap.servers", "kafka:9093")
            .option("subscribe", "store-transactions")
            .load())

# Transform binary data to string
store_transactions_df = store_transactions_df.selectExpr("CAST(value AS STRING)")

# Clean checkpoint
checkpoint_path = "/opt/spark/work-dir/checkpoints/"
dir_path = Path(checkpoint_path)
if dir_path.exists() and dir_path.is_dir():
    shutil.rmtree(dir_path)

# Schema alineado con faker_dataset.generate_record() y SparkUtils.generate_schema
store_transaction_schema = SparkUtils.generate_schema(
    [
        ("transaction_id", "string"),
        ("customer_id", "string"),
        ("customer_name", "string"),
        ("customer_email", "string"),
        ("customer_country", "string"),
        ("product_id", "string"),
        ("product_name", "string"),
        ("category", "string"),
        ("quantity", "int"),
        ("unit_price", "double"),
        ("total_amount", "double"),
        ("discount_pct", "double"),
        ("payment_method", "string"),
        ("status", "string"),
        ("order_date", "date"),
        ("order_timestamp", "timestamp"),
        ("shipping_country", "string"),
        ("shipping_city", "string"),
        ("warehouse_id", "int"),
        ("is_returned", "boolean"),
        ("review_score", "int"),
        ("review_text", "string"),
    ]
)

parsed_df = (
    store_transactions_df.withColumn(
        "data", F.from_json(F.col("value"), store_transaction_schema)
    ).select("data.*")
)

# Write stream in the destination
output_path = "/opt/spark/work-dir/data/streaming/output/"
query_events = (
    parsed_df.writeStream
    .outputMode("append")
    .format("parquet")
    .option("truncate", False)
    .option("checkpointLocation", checkpoint_path)
    .option("path", output_path)
    .partitionBy("warehouse_id")
    .start()
)

print("   Press Ctrl+C to stop.\n")
su.spark.streams.awaitAnyTermination()

26/05/10 23:04:52 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


   Press Ctrl+C to stop.



ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/opt/spark/python/lib/py4j-0.10.9.9-src.zip/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/opt/spark/python/lib/py4j-0.10.9.9-src.zip/py4j/clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
  File "/usr/lib/python3.10/socket.py", line 705, in readinto
    return self._sock.recv_into(b)
KeyboardInterrupt


KeyboardInterrupt: 

In [3]:
!ls -lah /opt/spark/work-dir/data/streaming/output/

total 0
drwxr-xr-x 1 root root 512 May 10 21:37  .
drwxrwxrwx 1 root root 512 May 10 21:34  ..
drwxr-xr-x 1 root root 512 May 10 21:37  _spark_metadata
drwxr-xr-x 1 root root 512 May 10 21:38 'warehouse_id=14'
drwxr-xr-x 1 root root 512 May 10 21:36 'warehouse_id=15'
drwxr-xr-x 1 root root 512 May 10 21:36 'warehouse_id=18'
drwxr-xr-x 1 root root 512 May 10 21:36 'warehouse_id=19'
drwxr-xr-x 1 root root 512 May 10 21:36 'warehouse_id=20'
drwxr-xr-x 1 root root 512 May 10 21:36 'warehouse_id=21'
drwxr-xr-x 1 root root 512 May 10 21:37 'warehouse_id=26'
drwxr-xr-x 1 root root 512 May 10 21:38 'warehouse_id=27'
drwxr-xr-x 1 root root 512 May 10 21:36 'warehouse_id=3'
drwxr-xr-x 1 root root 512 May 10 21:37 'warehouse_id=30'
drwxr-xr-x 1 root root 512 May 10 21:38 'warehouse_id=33'
drwxr-xr-x 1 root root 512 May 10 21:37 'warehouse_id=36'
drwxr-xr-x 1 root root 512 May 10 21:37 'warehouse_id=38'
drwxr-xr-x 1 root root 512 May 10 21:36 'warehouse_id=40'
drwxr-xr-x 1 root root 512 May 10 21:

In [4]:
su.spark.stop()